![Week 3 Day 4](../assets/banners/w3d4.png)

# Week 3 Day 4 - `Send`, Checkpoints, `interrupt()`, Subgraphs and Streaming

Yesterday you built your first graphs: state, nodes, edges, a reducer, and a check → repair loop that knows when to give up. Today you get LangGraph's power tools, the features that make it worth using over a hand-written loop:

- **`Send`**: mark every student's answer **in parallel**, however many there are (map-reduce)
- **Checkpointers**: save the graph after every step, rewind it (**time travel**), and survive a restart with SQLite
- **`interrupt()`**: pause the graph mid-run, wait for Asha, then carry on exactly where it stopped
- **Subgraphs** and **streaming**: compose graphs from graphs, and watch them work live

We'll use real data: five students' handwritten answers to Q3 (deadlocks), already transcribed (reading the handwriting is week 4!). And once again, **no API key needed**.

![Your 6-week roadmap](../assets/roadmap/week3.png)

### 🎯 By the end of this lab you will
1. 🗺️ Fan out with `Send` and fan in with a reducer (map-reduce)
2. 💾 Use `InMemorySaver` and `SqliteSaver`, read `get_state` and time-travel with `get_state_history`
3. ⏸️ Pause with `interrupt()` and resume with `Command(resume=...)`
4. 🪆 Use a compiled graph as a node (a subgraph)
5. 📡 Stream with `values`, `updates`, `custom` and `messages`, and know what LangGraph Studio is for

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | Day 4 - Parallel Map-Reduce with `Send` | 🧪 |
| 2 | Day 4 - Checkpointers and Human-in-the-Loop with `interrupt()` | 🧪 |
| 3 | Day 4 - Subgraphs, Streaming and LangGraph Studio | 🧪 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">This notebook writes one small file, <code>output/week3_day4_checkpoints.db</code>, to show checkpoints surviving a restart. <code>output/</code> is git-ignored, so it never ends up in your commits. Delete it any time; the notebook recreates it.</span>
        </td>
    </tr>
</table>

## Imports and the data

In [ ]:
import json
import operator
import sqlite3
from typing import Annotated, TypedDict
from uuid import uuid4

from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.config import get_stream_writer
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send, interrupt

from gradewise.config import DATA_DIR, OUTPUT_DIR
from gradewise.schemas import ExamPaper

exam = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
Q3 = next(q for q in exam.questions if q.id == "Q3")
transcripts = json.loads((DATA_DIR / "answer-sheets" / "synthetic" / "transcripts.json").read_text(encoding="utf-8"))
ANSWERS = {student: transcripts[student]["Q3"] for student in list(transcripts)[:5]}

print(Q3.text, f"[{Q3.max_marks:g} marks]\n")
for student, answer in ANSWERS.items():
    print(f"{student}: {answer[:90]!r}…")

## Part 1: Parallel map-reduce with `Send`

Yesterday's parallel nodes were **fixed**: you drew two edges from `START`. But Asha doesn't know in advance how many scripts she'll get: 5 today, 200 at the end of term. `Send` creates parallel tasks **at run time**:

![Diagram](../assets/diagrams/dc069236f916.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  S([START]) -- "Send × N" --> M1[mark_answer: student_01] & M2[mark_answer: student_02] & M3[mark_answer: …]
  M1 & M2 & M3 -- "results (reducer)" --> R[class_summary] --> E([END])
```

</details>


- The **map** step is a routing function that returns a list of `Send("node_name", private_input)`, one per task. Each task gets its *own* input, not the whole state.
- The **reduce** step works because `results` has a reducer, so all N updates are combined. (Remember yesterday's `InvalidUpdateError`?)

Our marker is a 🎭 stand-in for an LLM grader: keyword matching against the real Q3 criteria.

In [ ]:
from gradewise.display import show_graph

CONDITIONS = Q3.criteria[1].keywords  # ['mutual exclusion', 'hold and wait', 'no preemption', 'circular wait']


def keyword_mark_q3(answer: str) -> dict:
    """🎭 Stand-in grader for Q3: 2 marks for the definition, 1 per correct condition (max 2)."""
    text = answer.lower()
    definition_hits = [k for k in Q3.criteria[0].keywords if k in text]
    conditions = [c for c in CONDITIONS if c in text]
    definition = 2.0 if len(definition_hits) >= 2 else 1.0 if definition_hits else 0.0
    return {"definition": definition, "conditions": conditions, "awarded": definition + min(len(conditions), 2)}


class ClassState(TypedDict):
    answers: dict[str, str]  # student → answer
    results: Annotated[list[dict], operator.add]  # 💡 the reducer that makes fan-in work
    summary: str


class MarkTask(TypedDict):  # 💡 the PRIVATE input each parallel task receives
    student: str
    answer: str


def fan_out(state: ClassState) -> list[Send]:
    return [Send("mark_answer", {"student": s, "answer": a}) for s, a in state["answers"].items()]


def mark_answer(task: MarkTask) -> dict:
    return {"results": [{"student": task["student"], **keyword_mark_q3(task["answer"])}]}


def class_summary(state: ClassState) -> dict:
    marks = [r["awarded"] for r in state["results"]]
    return {
        "summary": f"{len(marks)} scripts · average {sum(marks) / len(marks):.1f}/{Q3.max_marks:g} · top {max(marks):g}"
    }


def build_marker(checkpointer=None):
    builder = StateGraph(ClassState)
    builder.add_node("mark_answer", mark_answer)
    builder.add_node("class_summary", class_summary)
    builder.add_conditional_edges(START, fan_out, ["mark_answer"])  # 🗺️ map
    builder.add_edge("mark_answer", "class_summary")  # 🧮 reduce
    builder.add_edge("class_summary", END)
    return builder.compile(checkpointer=checkpointer)


marker = build_marker()
show_graph(marker)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">We're about to mark 5 answers. How many times will <code>class_summary</code> run: once, or five times? (Hint: all five <code>mark_answer</code> tasks run in the <b>same superstep</b>, and <code>class_summary</code> only runs in the next one.)</span>
        </td>
    </tr>
</table>

In [ ]:
result = marker.invoke({"answers": ANSWERS, "results": []})

for r in result["results"]:
    detail = f"definition {r['definition']:g}, conditions {r['conditions']}"
    print(f"{r['student']}: {r['awarded']:g}/{Q3.max_marks:g}  ({detail})")
print("\n📊", result["summary"])

💡 Look at `student_03`: "Starvation" isn't one of the four conditions, so it earns nothing. And `student_05` wrote "programs" instead of "processes", so the definition earned only partial credit. A keyword marker is crude; an LLM grader will be smarter, but the **graph shape stays exactly the same**. That's the point: you can design and test the orchestration for free, then drop the model in.

## Part 2: Checkpointers and time travel

Compile with a **checkpointer** and LangGraph saves a **snapshot of the state after every superstep**, filed under a `thread_id`. That one feature unlocks memory (day 2), pause-and-resume (Part 3), crash recovery, and time travel.

In [ ]:
marker = build_marker(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "cs201-q3-batch1"}}
marker.invoke({"answers": ANSWERS, "results": []}, config)

snapshot = marker.get_state(config)  # 💡 the LATEST checkpoint
print("Summary in the saved state:", snapshot.values["summary"])
print("Next node(s) to run:", snapshot.next, "← empty: the run finished")

### `get_state_history`: every step, newest first

In [ ]:
history = list(marker.get_state_history(config))
for snap in history:
    print(f"step {snap.metadata['step']:>2} · next={str(snap.next):45} · {len(snap.values.get('results', []))} results")

### ⏪ Time travel: replay and fork
Every snapshot has its own `config` (with a `checkpoint_id`). Pass it back to resume **from that moment**:
- **Replay:** `invoke(None, snap.config)` re-runs from there. Steps before it are *not* re-run.
- **Fork:** `update_state(snap.config, changes, as_node=...)` writes a new checkpoint branching off that point, as if `as_node` had produced `changes`. Then `invoke(None, new_config)` continues down the new branch.

Scenario: a **late script** from `student_06` arrives after marking. Instead of re-marking everyone, Asha forks the run just before `class_summary` and slips the extra result in:

In [ ]:
before_summary = next(s for s in history if s.next == ("class_summary",))

late = {"student": "student_06", **keyword_mark_q3(transcripts["student_06"]["Q3"])}
fork_config = marker.update_state(before_summary.config, {"results": [late]}, as_node="mark_answer")
forked = marker.invoke(None, fork_config)

print("Original run:", history[0].values["summary"])
print("Forked run:  ", forked["summary"])
print(f"The thread now has {len(list(marker.get_state_history(config)))} checkpoints (the old branch is still there)")

### 💾 `SqliteSaver`: checkpoints that survive a restart
`InMemorySaver` forgets everything when the kernel restarts. `SqliteSaver` writes to a file. We'll save a run, then build a **brand-new** saver and graph (think: Python restarted) and read the state back.

In [ ]:
db_path = OUTPUT_DIR / "week3_day4_checkpoints.db"
db_path.parent.mkdir(parents=True, exist_ok=True)

# ⚠️ check_same_thread=False: LangGraph may run nodes on another thread than the one that opened the file
first = build_marker(SqliteSaver(sqlite3.connect(str(db_path), check_same_thread=False)))
# ⚠️ The file outlives this notebook, so each run gets a NEW thread: re-using a finished thread would
#    add this run's results to last run's (the reducer!), and you'd see 10 scripts instead of 5.
sqlite_config = {"configurable": {"thread_id": f"batch-{uuid4().hex[:6]}"}}
first.invoke({"answers": ANSWERS, "results": []}, sqlite_config)

# 🔄 "Restart": a fresh connection, a fresh graph object, the same file
second = build_marker(SqliteSaver(sqlite3.connect(str(db_path), check_same_thread=False)))
print("Read back after 'restart':", second.get_state(sqlite_config).values["summary"])

## Part 3: Human-in-the-loop with `interrupt()`

Some decisions belong to a human. Inside any node, `interrupt(payload)` **pauses the whole graph**: the checkpointer saves everything and `invoke` returns with an `"__interrupt__"` key holding your payload. Later, `invoke(Command(resume=answer), config)` carries on, and inside the node, `interrupt()` now **returns** `answer`.

![Diagram](../assets/diagrams/e9794fb6e753.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
sequenceDiagram
  participant Code as Your code
  participant G as Graph
  participant A as Asha
  Code->>G: invoke(input, config)
  G-->>Code: {"__interrupt__": [payload]} ⏸ saved
  Code->>A: show the flagged scripts
  A-->>Code: {"student_05": 2}
  Code->>G: invoke(Command(resume={...}), config)
  G-->>Code: final state ▶️
```

</details>


Let's add a `moderation` node: any script where the definition earned only *partial* credit goes to Asha before the summary.

In [ ]:
class ModeratedState(ClassState):
    overrides: dict[str, float]  # student → Asha's mark (no reducer: overwrite)


def moderation(state: ModeratedState) -> dict:
    flagged = {r["student"]: r["awarded"] for r in state["results"] if r["definition"] == 1.0}
    print(f"▶️ moderation starts ({len(flagged)} flagged)")  # 💡 watch: this prints TWICE
    if not flagged:
        return {"overrides": {}}
    decision = interrupt({"question": "Q3", "flagged": flagged, "answers": {s: state["answers"][s] for s in flagged}})
    print("✅ Asha replied:", decision)
    return {"overrides": decision}


def moderated_summary(state: ModeratedState) -> dict:
    marks = {r["student"]: r["awarded"] for r in state["results"]} | state.get("overrides", {})
    return {"summary": " · ".join(f"{s[-2:]}: {m:g}" for s, m in sorted(marks.items()))}


builder = StateGraph(ModeratedState)
builder.add_node("mark_answer", mark_answer)
builder.add_node("moderation", moderation)
builder.add_node("class_summary", moderated_summary)
builder.add_conditional_edges(START, fan_out, ["mark_answer"])
builder.add_edge("mark_answer", "moderation")
builder.add_edge("moderation", "class_summary")
builder.add_edge("class_summary", END)
moderated = builder.compile(checkpointer=InMemorySaver())  # ⚠️ interrupt() NEEDS a checkpointer

config = {"configurable": {"thread_id": "moderation-1"}}
paused = moderated.invoke({"answers": ANSWERS, "results": []}, config)

In [ ]:
payload = paused["__interrupt__"][0].value
print("⏸ Paused. Asha is asked about:", payload["flagged"])
for student, answer in payload["answers"].items():
    print(f"   {student}: {answer!r}")
print("\nThe graph is waiting at:", moderated.get_state(config).next)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">When we resume, which line of <code>moderation</code> runs first: the one after <code>interrupt()</code>, or the <code>print("▶️ moderation starts…")</code> at the top? Make your guess, then run the next cell.</span>
        </td>
    </tr>
</table>

In [ ]:
# 🧑‍🏫 Asha: "student_04 explains it well enough for full marks; student_05 keeps 3."
final = moderated.invoke(Command(resume={"student_04": 4.0, "student_05": 3.0}), config)
print("\n📊", final["summary"])

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">On resume, LangGraph re-runs the interrupted node <b>from the top</b>; this time <code>interrupt()</code> returns the answer instead of pausing. That's why "▶️ moderation starts" printed twice. So: <b>never put side effects before <code>interrupt()</code></b> (no saving to a database, no emails, no paid API calls), and keep the interrupt payload JSON-serialisable so any checkpointer can save it.</span>
        </td>
    </tr>
</table>

💡 Two more tools in the same family, for later:
| Tool | Use it when |
|---|---|
| `interrupt_before=["node"]` / `interrupt_after=["node"]` in `compile()` | You want a static breakpoint for debugging, without editing node code |
| A node returning `Command(update={...}, goto="node")` | A node wants to update state **and** choose the next node itself, without a separate routing function |

## Part 4: Subgraphs

A compiled graph is itself a runnable, so you can use it **as a node** in a bigger graph. Teams build and test parts separately, then compose them. When parent and child share state keys, just add the compiled child with `add_node`. Here's a tiny "report card" subgraph turning a mark into a grade band and feedback:

In [ ]:
from gradewise.display import show_graph

class ReportState(TypedDict):
    awarded: float
    band: str
    feedback: str


def grade_band(state: ReportState) -> dict:
    pct = state["awarded"] / Q3.max_marks
    return {"band": "A" if pct >= 0.85 else "B" if pct >= 0.7 else "C" if pct >= 0.5 else "D"}


def write_feedback(state: ReportState) -> dict:
    return {"feedback": "Excellent: all four marks." if state["band"] == "A" else "Revise the four Coffman conditions."}


sub = StateGraph(ReportState)
sub.add_node("grade_band", grade_band)
sub.add_node("write_feedback", write_feedback)
sub.add_edge(START, "grade_band")
sub.add_edge("grade_band", "write_feedback")
sub.add_edge("write_feedback", END)
report_card = sub.compile()


class StudentState(TypedDict):
    student: str
    answer: str
    awarded: float  # 💡 shared with the subgraph
    band: str
    feedback: str


parent = StateGraph(StudentState)
parent.add_node("mark", lambda s: {"awarded": keyword_mark_q3(s["answer"])["awarded"]})
parent.add_node("report_card", report_card)  # 🪆 a whole graph, used as one node
parent.add_edge(START, "mark")
parent.add_edge("mark", "report_card")
parent.add_edge("report_card", END)
student_graph = parent.compile()

show_graph(student_graph, xray=True)  # 💡 xray=True draws inside the subgraph
student_graph.invoke({"student": "student_03", "answer": ANSWERS["student_03"]})

## Part 5: Streaming

`invoke` waits for the end. `stream` yields as the graph runs, and `stream_mode` chooses *what* you get:

| `stream_mode` | Each chunk is | Good for |
|---|---|---|
| `"values"` | the **full state** after every superstep | debugging, simple UIs |
| `"updates"` | `{node: its update}` after each node | progress bars, logs |
| `"custom"` | anything a node sends with `get_stream_writer()` | "Marking student 3 of 5…" messages |
| `"messages"` | `(token, metadata)` from any LLM called inside a node | typing-as-it-thinks chat UIs |
| `"debug"` | everything, in detail | deep debugging |

You can ask for several at once with a list, and add `subgraphs=True` to see inside subgraphs.

In [ ]:
print("── updates ──")
for chunk in marker.stream(
    {"answers": ANSWERS, "results": []}, {"configurable": {"thread_id": "stream-u"}}, stream_mode="updates"
):
    print(chunk)

print("\n── values ── (the state fills in, one superstep at a time)")
for chunk in student_graph.stream({"student": "student_01", "answer": ANSWERS["student_01"]}, stream_mode="values"):
    print({k: v for k, v in chunk.items() if k != "answer"})

In [ ]:
def mark_with_progress(task: MarkTask) -> dict:
    writer = get_stream_writer()  # 💡 send your own progress events
    writer({"progress": f"🖊️ marking {task['student']}…"})
    return mark_answer(task)


builder = StateGraph(ClassState)
builder.add_node("mark_answer", mark_with_progress)
builder.add_node("class_summary", class_summary)
builder.add_conditional_edges(START, fan_out, ["mark_answer"])
builder.add_edge("mark_answer", "class_summary")
builder.add_edge("class_summary", END)

for mode, chunk in builder.compile().stream({"answers": ANSWERS, "results": []}, stream_mode=["custom", "updates"]):
    print(f"{mode:7} {chunk if mode == 'custom' else list(chunk)}")

And `"messages"` mode streams **tokens** from any chat model a node calls. A 🎭 stand-in chat model is enough to see it (it "streams" word by word):

In [ ]:
feedback_llm = GenericFakeChatModel(
    messages=iter([AIMessage(content="Good definition! Next time, name two of the four Coffman conditions.")])
)


def llm_feedback(state: StudentState) -> dict:
    return {"feedback": feedback_llm.invoke(f"Write feedback for: {state['answer']}").content}


builder = StateGraph(StudentState)
builder.add_node("llm_feedback", llm_feedback)
builder.add_edge(START, "llm_feedback")
builder.add_edge("llm_feedback", END)

for token, metadata in builder.compile().stream(
    {"student": "student_03", "answer": ANSWERS["student_03"]}, stream_mode="messages"
):
    print(token.content, end="", flush=True)
    node = metadata["langgraph_node"]  # 💡 the metadata says which node the tokens came from
print(f"\n(from node: {node})")

### 🎨 LangGraph Studio
**LangGraph Studio** is a visual IDE for graphs: you see your graph drawn live, step through runs, inspect and **edit the state at any checkpoint** (time travel with a mouse), and answer interrupts from a form. It runs your graph through a local dev server described by a `langgraph.json` file at the project root:

```json
{
  "dependencies": ["."],
  "graphs": {"exam": "./gradewise/exam/graph.py:build_exam_graph"},
  "env": ".env"
}
```

Then start it with the LangGraph CLI (it's optional, so it isn't in the course's `pyproject.toml`): `uvx --from "langgraph-cli[inmem]" langgraph dev`. It opens Studio in your browser and needs a free LangSmith account. Everything you did in this notebook (state, checkpoints, interrupts) is exactly what Studio shows you visually.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">Checkpoints turn "the AI crashed halfway through 200 scripts" from a disaster into a non-event: resume from the last checkpoint and pay only for what's left. And <code>interrupt()</code> is how you put a named, accountable human at exactly the point where the stakes are highest. In regulated industries, an <b>audit trail</b> (the full state history) plus <b>human sign-off</b> is often the difference between "interesting prototype" and "approved for production".</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Everything today appears in GradeWise. <code>Send</code> fans out one grading task per question in week 5's orchestrator. <code>gradewise/exam/graph.py</code> uses <code>interrupt()</code> in <code>admin_review</code> and <code>SqliteSaver</code> via <code>sqlite_checkpointer()</code>, writing to <code>output/exam_checkpoints.db</code>. Open it now and find the ⚠️ comment about side effects before <code>interrupt()</code>: it's today's lesson.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 🗺️ marked 5 real Q3 answers in parallel with `Send`, fanning back in through a reducer
- 💾 saved every step with a checkpointer, read `get_state` and `get_state_history`, then **replayed and forked** a run
- 🔄 survived a "restart" with `SqliteSaver`
- ⏸️ paused for Asha with `interrupt()` and resumed with `Command(resume=...)`, and saw the node re-run from the top
- 🪆 used a compiled graph as a node, and 📡 streamed `updates`, `values`, `custom` and `messages`

**Tomorrow (Day 5):** 🧱 project day! All of this week comes together in GradeWise's **exam understanding v2**: extract → validate → retry → admin approval → store.

⏭ **Next:** `week3/day5.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Add a <b>"suspiciously short"</b> check to the map-reduce marker. Give the state a second reducer key, <code>flags: Annotated[list[str], operator.add]</code>, and make <code>mark_answer</code> flag any answer with fewer than 30 words. After marking, a <code>review</code> node should <code>interrupt()</code> <b>only if there are flags</b>, showing Asha the short answers; she resumes with a dict of mark overrides, which the summary applies. Use a <code>SqliteSaver</code>, and prove that a brand-new graph object can resume the paused thread. Solution: <code>week3/solutions/day4_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• Map-reduce with <code>Send</code>: <a href="https://docs.langchain.com/oss/python/langgraph/graph-api#send">docs.langchain.com/oss/python/langgraph/graph-api#send</a><br/> • Persistence and checkpointers: <a href="https://docs.langchain.com/oss/python/langgraph/persistence">docs.langchain.com/oss/python/langgraph/persistence</a><br/> • Time travel: <a href="https://docs.langchain.com/oss/python/langgraph/use-time-travel">docs.langchain.com/oss/python/langgraph/use-time-travel</a><br/> • Interrupts: <a href="https://docs.langchain.com/oss/python/langgraph/interrupts">docs.langchain.com/oss/python/langgraph/interrupts</a><br/> • Subgraphs and streaming: <a href="https://docs.langchain.com/oss/python/langgraph/use-subgraphs">use-subgraphs</a> · <a href="https://docs.langchain.com/oss/python/langgraph/streaming">streaming</a><br/> • LangGraph Studio: <a href="https://docs.langchain.com/langsmith/studio">docs.langchain.com/langsmith/studio</a></span>
        </td>
    </tr>
</table>